# PCA in a Machine-Learning Pipeline

Companion notebook for Sections 4.3, 4.7, and 8 of the lecture notes.

Objectives:

- use PCA inside a scikit-learn `Pipeline`;
- select `pca__n_components` with cross-validation;
- avoid data leakage when scaling and reducing dimensionality;
- show why explained variance is not the same as predictive relevance;
- compare PCA with and without a downstream validation criterion.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True


## 1. Correct pipeline pattern

Scaling and PCA both learn parameters from data. Inside cross-validation, they must be fit only on each training fold.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('clf', LogisticRegression(max_iter=2000)),
])

param_grid = {
    'pca__n_components': [5, 10, 20, 30, 40, 0.90, 0.95],
    'clf__C': [0.1, 1.0, 10.0],
}

search = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
search.fit(X, y)

print('Best parameters:', search.best_params_)
print('Best CV accuracy:', round(search.best_score_, 4))


In [ ]:
results = pd.DataFrame(search.cv_results_)
cols = ['param_pca__n_components', 'param_clf__C', 'mean_test_score', 'std_test_score', 'rank_test_score']
results[cols].sort_values('rank_test_score').head(10)


## 2. Validation performance across component counts

Explained variance is useful, but a predictive model can prefer a different value of `k`.


In [ ]:
rows = []
for k in [2, 5, 10, 20, 30, 40, 0.90, 0.95, None]:
    steps = [('scaler', StandardScaler())]
    label = 'no PCA' if k is None else str(k)
    if k is not None:
        steps.append(('pca', PCA(n_components=k, svd_solver='full')))
    steps.append(('clf', KNeighborsClassifier(n_neighbors=5)))
    model = Pipeline(steps)
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    rows.append({'n_components': label, 'accuracy_mean': scores.mean(), 'accuracy_std': scores.std()})

knn_results = pd.DataFrame(rows)
knn_results


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
labels = knn_results['n_components'].astype(str)
ax.errorbar(labels, knn_results['accuracy_mean'], yerr=knn_results['accuracy_std'], marker='o', capsize=3)
ax.set_xlabel('PCA n_components')
ax.set_ylabel('5-fold CV accuracy')
ax.set_title('Downstream performance can guide component selection')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 3. Leakage demonstration

The correct pattern fits preprocessing inside the cross-validation loop. The leaky pattern fits scaling and PCA once on the full dataset and only then cross-validates the classifier. The numerical difference may be small for PCA, but the second pattern is still invalid because validation-fold information influenced the preprocessing transformation.


In [ ]:
correct_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=30)),
    ('clf', LogisticRegression(max_iter=2000)),
])
correct_scores = cross_val_score(correct_pipe, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

# Leaky preprocessing: scaler and PCA see the full dataset before CV splits are evaluated.
X_scaled_full = StandardScaler().fit_transform(X)
X_pca_full = PCA(n_components=30).fit_transform(X_scaled_full)
leaky_scores = cross_val_score(LogisticRegression(max_iter=2000), X_pca_full, y, cv=cv, scoring='accuracy', n_jobs=-1)

pd.DataFrame({
    'pattern': ['correct Pipeline', 'leaky precomputed PCA'],
    'accuracy_mean': [correct_scores.mean(), leaky_scores.mean()],
    'accuracy_std': [correct_scores.std(), leaky_scores.std()],
})


## 4. High variance is not predictive relevance

The next synthetic dataset has one high-variance direction that is mostly irrelevant and one low-variance direction that carries the class signal. PCA with one component keeps the high-variance direction and can hurt classification.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(3)
n = 1200
y_syn = rng.integers(0, 2, size=n)

x_high_variance = rng.normal(0, 8.0, size=n)
x_low_variance_signal = (2 * y_syn - 1) * 0.45 + rng.normal(0, 0.18, size=n)
X_syn = np.column_stack([x_high_variance, x_low_variance_signal])

X_train, X_test, y_train, y_test = train_test_split(X_syn, y_syn, test_size=0.3, stratify=y_syn, random_state=42)

models = {
    'LogReg on original features': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())]),
    'PCA(1) + LogReg': Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=1)), ('clf', LogisticRegression())]),
    'PCA(2) + LogReg': Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=2)), ('clf', LogisticRegression())]),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({'model': name, 'test_accuracy': accuracy_score(y_test, pred)})

pd.DataFrame(rows)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_syn[:, 0], X_syn[:, 1], c=y_syn, cmap='coolwarm', s=12, alpha=0.45)
ax.set_xlabel('High-variance mostly irrelevant feature')
ax.set_ylabel('Low-variance predictive feature')
ax.set_title('The predictive direction can have low variance')
plt.show()

pca_syn = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=2))])
pca_syn.fit(X_syn)
print('Explained variance ratio:', pca_syn.named_steps['pca'].explained_variance_ratio_)


## Exercises

1. Change the classifier from logistic regression to k-NN in the grid search. Does the best `n_components` change?
2. Repeat the leakage comparison with `n_components=0.95`. Is the difference larger or smaller?
3. In the synthetic dataset, increase the noise in the low-variance signal. When does PCA(1) become especially harmful?


## Takeaways

- Fit scaling and PCA inside the pipeline, not before the validation split.
- `pca__n_components` can be selected with cross-validation.
- Explained variance measures reconstruction/compression quality, not necessarily supervised predictive utility.
